# 08 Representation Redundancy

**Purpose.** Representational similarity across image variants and encoders.

**Thesis reference.** §5 Representation Comparison

**Inputs / outputs.** See the data-layout table in the repository README. Paths near the top of the notebook (e.g. `PROFILE_DIR`, `IMAGE_ROOT`, `LABELS`) are relative to the repository root and assume the folder layout described there.

---

## Configuration

In [ ]:
IMAGE_ROOT           = "images224"
LABELS_FILE          = "labels/labels_main_any_ckd.json"
REDUNDANCY_THRESHOLD = 0.90    # RDM correlation above which two are redundant
MAX_PATIENTS         = None    # None = all; set e.g. 60 for a quick check

# When two representations are redundant, keep the one listed first here.
# Grayscale leads because it is the simplest and is what the draft paper
# reported, so keeping it preserves comparability with the earlier results.
PREFERENCE_ORDER = [
    "grayscale_heatmap",
    "rgb_mz_split",
    "tic_plot",
    "recurrence_plot",
    "lineplot_intensity",
    "pseudocolor_heatmap",
    "scatterplot",
]


## Code

In [ ]:
import json
from itertools import combinations
from pathlib import Path

import numpy as np
from PIL import Image
from scipy.spatial.distance import pdist
from scipy.stats import spearmanr

REPRESENTATIONS = [
    "grayscale_heatmap", "pseudocolor_heatmap", "rgb_mz_split",
    "scatterplot", "tic_plot", "lineplot_intensity", "recurrence_plot",
]


def load_representation(root, rep, pids):
    """Load all patient images for one representation as a flat matrix."""
    rows = []
    for pid in pids:
        im = Image.open(Path(root) / rep / f"{pid}.png")
        a = np.asarray(im, dtype=np.float32).ravel() / 255.0
        rows.append(a)
    return np.stack(rows)


def rdm(X):
    """Upper triangle of the patient-by-patient correlation-distance matrix."""
    Xc = X - X.mean(axis=1, keepdims=True)
    sd = Xc.std(axis=1, keepdims=True)
    sd[sd == 0] = 1.0
    return pdist(Xc / sd, metric="correlation")


def main():
    root = Path(IMAGE_ROOT)
    avail = [r for r in REPRESENTATIONS if (root / r).is_dir()]
    if not avail:
        raise SystemExit(f"No representation folders found in {root}")

    pids = sorted(p.stem for p in (root / avail[0]).glob("*.png"))
    for r in avail[1:]:
        pids = [p for p in pids if (root / r / f"{p}.png").exists()]
    if MAX_PATIENTS:
        pids = pids[:MAX_PATIENTS]
    print(f"Representations: {len(avail)}")
    print(f"Patients: {len(pids)}\n")

    labels = None
    if Path(LABELS_FILE).exists():
        with open(LABELS_FILE) as fh:
            obj = json.load(fh)
        lab = obj.get("labels", obj)
        if all(p in lab for p in pids):
            labels = np.array([lab[p] for p in pids])

    # build one RDM per representation
    rdms, sep = {}, {}
    for rep in avail:
        X = load_representation(root, rep, pids)
        rdms[rep] = rdm(X)
        if labels is not None:
            Xc = X - X.mean(0)
            yc = labels - labels.mean()
            denom = (Xc.std(0) * yc.std())
            denom[denom == 0] = 1.0
            r = (Xc * yc[:, None]).mean(0) / denom
            sep[rep] = float(np.nanmax(np.abs(r)))
        print(f"  {rep:22s} RDM built ({len(rdms[rep]):,} pairs)"
              + (f"   max|r| with label = {sep[rep]:.3f}" if sep else ""))

    # correlate the RDMs
    n = len(avail)
    M = np.eye(n)
    print("\nRDM correlation (Spearman); high means redundant\n")
    for i, j in combinations(range(n), 2):
        rho = spearmanr(rdms[avail[i]], rdms[avail[j]]).statistic
        M[i, j] = M[j, i] = rho

    w = max(len(r) for r in avail) + 1
    print(" " * w + "".join(f"{r[:9]:>10s}" for r in avail))
    for i, r in enumerate(avail):
        print(f"{r:<{w}}" + "".join(f"{M[i, j]:10.3f}" for j in range(n)))

    # greedy minimal subset
    print(f"\nRedundant pairs (rho > {REDUNDANCY_THRESHOLD}):")
    pairs = [(avail[i], avail[j], M[i, j])
             for i, j in combinations(range(n), 2)
             if M[i, j] > REDUNDANCY_THRESHOLD]
    if pairs:
        for a, b, rho in sorted(pairs, key=lambda t: -t[2]):
            print(f"  {a:22s} <-> {b:22s}  {rho:.3f}")
    else:
        print("  none; all representations carry distinct geometry")

    keep, dropped = [], {}
    order = sorted(avail, key=lambda r: (PREFERENCE_ORDER.index(r)
                                         if r in PREFERENCE_ORDER else 99))
    for rep in order:
        i = avail.index(rep)
        clash = None
        for k in keep:
            if M[i, avail.index(k)] > REDUNDANCY_THRESHOLD:
                clash = k
                break
        if clash:
            dropped[rep] = clash
        else:
            keep.append(rep)

    print(f"\nSuggested subset ({len(keep)} of {len(avail)}):")
    for r in keep:
        print(f"  KEEP  {r}")
    for r, because in dropped.items():
        print(f"  drop  {r:22s} (redundant with {because})")

    saving = 1 - len(keep) / len(avail)
    print(f"\nEmbedding extraction cost reduced by {saving:.0%}.")

    with open("redundancy_results.json", "w") as fh:
        json.dump({
            "representations": avail,
            "n_patients":      len(pids),
            "rdm_correlation": M.tolist(),
            "separability":    sep,
            "threshold":       REDUNDANCY_THRESHOLD,
            "keep":            keep,
            "dropped":         dropped,
        }, fh, indent=2)
    print("Written to redundancy_results.json")


main()


## Run

In [ ]:
main()

In [7]:
IMAGE_ROOT           = "images224"
LABELS_FILE          = "labels/labels_main_any_ckd.json"
REDUNDANCY_THRESHOLD = 0.90    # RDM correlation above which two are redundant
MAX_PATIENTS         = None    # None = all; set e.g. 60 for a quick check

# When two representations are redundant, keep the one listed first here.
# Grayscale leads because it is the simplest and is what the draft paper
# reported, so keeping it preserves comparability with the earlier results.
PREFERENCE_ORDER = [
    "grayscale_heatmap",
    "rgb_mz_split",
    "tic_plot",
    "recurrence_plot",
    "lineplot_intensity",
    "pseudocolor_heatmap",
    "scatterplot",
]


In [5]:
import json
from itertools import combinations
from pathlib import Path

import numpy as np
from PIL import Image
from scipy.spatial.distance import pdist
from scipy.stats import spearmanr

REPRESENTATIONS = [
    "grayscale_heatmap", "pseudocolor_heatmap", "rgb_mz_split",
    "scatterplot", "tic_plot", "lineplot_intensity", "recurrence_plot",
]


def load_representation(root, rep, pids):
    """Load all patient images for one representation as a flat matrix."""
    rows = []
    for pid in pids:
        im = Image.open(Path(root) / rep / f"{pid}.png")
        a = np.asarray(im, dtype=np.float32).ravel() / 255.0
        rows.append(a)
    return np.stack(rows)


def rdm(X):
    """Upper triangle of the patient-by-patient correlation-distance matrix."""
    Xc = X - X.mean(axis=1, keepdims=True)
    sd = Xc.std(axis=1, keepdims=True)
    sd[sd == 0] = 1.0
    return pdist(Xc / sd, metric="correlation")


def main():
    root = Path(IMAGE_ROOT)
    avail = [r for r in REPRESENTATIONS if (root / r).is_dir()]
    if not avail:
        raise SystemExit(f"No representation folders found in {root}")

    pids = sorted(p.stem for p in (root / avail[0]).glob("*.png"))
    for r in avail[1:]:
        pids = [p for p in pids if (root / r / f"{p}.png").exists()]
    if MAX_PATIENTS:
        pids = pids[:MAX_PATIENTS]
    print(f"Representations: {len(avail)}")
    print(f"Patients: {len(pids)}\n")

    labels = None
    if Path(LABELS_FILE).exists():
        with open(LABELS_FILE) as fh:
            obj = json.load(fh)
        lab = obj.get("labels", obj)
        if all(p in lab for p in pids):
            labels = np.array([lab[p] for p in pids])

    # build one RDM per representation
    rdms, sep = {}, {}
    for rep in avail:
        X = load_representation(root, rep, pids)
        rdms[rep] = rdm(X)
        if labels is not None:
            Xc = X - X.mean(0)
            yc = labels - labels.mean()
            denom = (Xc.std(0) * yc.std())
            denom[denom == 0] = 1.0
            r = (Xc * yc[:, None]).mean(0) / denom
            sep[rep] = float(np.nanmax(np.abs(r)))
        print(f"  {rep:22s} RDM built ({len(rdms[rep]):,} pairs)"
              + (f"   max|r| with label = {sep[rep]:.3f}" if sep else ""))

    # correlate the RDMs
    n = len(avail)
    M = np.eye(n)
    print("\nRDM correlation (Spearman); high means redundant\n")
    for i, j in combinations(range(n), 2):
        rho = spearmanr(rdms[avail[i]], rdms[avail[j]]).statistic
        M[i, j] = M[j, i] = rho

    w = max(len(r) for r in avail) + 1
    print(" " * w + "".join(f"{r[:9]:>10s}" for r in avail))
    for i, r in enumerate(avail):
        print(f"{r:<{w}}" + "".join(f"{M[i, j]:10.3f}" for j in range(n)))

    # greedy minimal subset
    print(f"\nRedundant pairs (rho > {REDUNDANCY_THRESHOLD}):")
    pairs = [(avail[i], avail[j], M[i, j])
             for i, j in combinations(range(n), 2)
             if M[i, j] > REDUNDANCY_THRESHOLD]
    if pairs:
        for a, b, rho in sorted(pairs, key=lambda t: -t[2]):
            print(f"  {a:22s} <-> {b:22s}  {rho:.3f}")
    else:
        print("  none; all representations carry distinct geometry")

    keep, dropped = [], {}
    order = sorted(avail, key=lambda r: (PREFERENCE_ORDER.index(r)
                                         if r in PREFERENCE_ORDER else 99))
    for rep in order:
        i = avail.index(rep)
        clash = None
        for k in keep:
            if M[i, avail.index(k)] > REDUNDANCY_THRESHOLD:
                clash = k
                break
        if clash:
            dropped[rep] = clash
        else:
            keep.append(rep)

    print(f"\nSuggested subset ({len(keep)} of {len(avail)}):")
    for r in keep:
        print(f"  KEEP  {r}")
    for r, because in dropped.items():
        print(f"  drop  {r:22s} (redundant with {because})")

    saving = 1 - len(keep) / len(avail)
    print(f"\nEmbedding extraction cost reduced by {saving:.0%}.")

    with open("redundancy_results.json", "w") as fh:
        json.dump({
            "representations": avail,
            "n_patients":      len(pids),
            "rdm_correlation": M.tolist(),
            "separability":    sep,
            "threshold":       REDUNDANCY_THRESHOLD,
            "keep":            keep,
            "dropped":         dropped,
        }, fh, indent=2)
    print("Written to redundancy_results.json")


main()


In [8]:
main()

Representations: 7
Patients: 597

  grayscale_heatmap      RDM built (177,906 pairs)
  pseudocolor_heatmap    RDM built (177,906 pairs)
  rgb_mz_split           RDM built (177,906 pairs)
  scatterplot            RDM built (177,906 pairs)
  tic_plot               RDM built (177,906 pairs)
  lineplot_intensity     RDM built (177,906 pairs)
  recurrence_plot        RDM built (177,906 pairs)

RDM correlation (Spearman) --- high means redundant

                     grayscale pseudocol rgb_mz_sp scatterpl  tic_plot lineplot_ recurrenc
grayscale_heatmap        1.000     0.209     0.148    -0.092     0.300     0.421     0.003
pseudocolor_heatmap      0.209     1.000    -0.109     0.057     0.252     0.153    -0.004
rgb_mz_split             0.148    -0.109     1.000    -0.020     0.152     0.184     0.000
scatterplot             -0.092     0.057    -0.020     1.000    -0.021    -0.083     0.002
tic_plot                 0.300     0.252     0.152    -0.021     1.000     0.712     0.005
lineplot_